# Visualiser les Boundary Tokens - Correctement
Avec le tokenizer exact du modèle

In [ ]:
from google.colab import drive
import os, time
drive.mount('/content/drive')
time.sleep(2)
os.chdir('/content/drive/MyDrive/khabar-segmentation')
print(f"Working dir: {os.getcwd()}")

In [ ]:
!pip install transformers -q
print("OK")

In [ ]:
import json
from transformers import AutoTokenizer
from pathlib import Path

# Load corpus
print("[1/5] Load corpus...")
with open('data/processed/kitab_uqala_reference_corpus.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print(f"      {len(text):,} chars")

# Load boundary tokens JSON
print("\n[2/5] Load boundary tokens JSON...")
with open('results/camelbert_boundary_tokens_clean.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

boundary_indices = set(data['boundary_indices'])
metadata = data['metadata']
print(f"      {len(boundary_indices):,} boundary token indices")

# Load tokenizer
print("\n[3/5] Load tokenizer...")
tokenizer = AutoTokenizer.from_pretrained('CAMeL-Lab/bert-base-arabic-camelbert-msa')
print(f"      OK")

In [ ]:
# Tokenize corpus
print("\n[4/5] Tokenize corpus...")
encoded = tokenizer(
    text,
    return_tensors='pt',
    return_offsets_mapping=True,
    truncation=False,
    padding=False,
)

token_ids = encoded['input_ids'][0].numpy()
offsets = encoded['offset_mapping'][0].numpy()

print(f"      {len(token_ids):,} tokens")
print(f"      First 20 tokens:")
for i in range(min(20, len(token_ids))):
    token_text = tokenizer.decode([token_ids[i]])
    char_start, char_end = offsets[i]
    in_boundary = i in boundary_indices
    marker = "*" if in_boundary else " "
    print(f"        {i:5d}{marker} {token_text:15s} [{char_start:5d}:{char_end:5d}]")

In [ ]:
# Build HTML with character-level highlighting
print("\n[5/5] Generate HTML...")

# Create a map of character positions to highlight
char_positions_to_highlight = set()

for token_idx in boundary_indices:
    if token_idx < len(offsets):
        char_start, char_end = offsets[token_idx]
        for char_pos in range(char_start, char_end):
            char_positions_to_highlight.add(char_pos)

print(f"      Character positions to highlight: {len(char_positions_to_highlight):,}")

# Build HTML text
html_text = ""
for char_idx, char in enumerate(text):
    if char_idx in char_positions_to_highlight:
        html_text += f'<span class="boundary">{char}</span>'
    else:
        # Escape HTML
        if char == '&':
            html_text += '&amp;'
        elif char == '<':
            html_text += '&lt;'
        elif char == '>':
            html_text += '&gt;'
        else:
            html_text += char

print(f"      HTML text built")

In [ ]:
# Create HTML page
html = f"""<!DOCTYPE html>
<html dir="rtl" lang="ar">
<head>
    <meta charset="UTF-8">
    <title>Boundary Tokens - Kitab Uqala</title>
    <link href="https://fonts.googleapis.com/css2?family=Noto+Naskh+Arabic:wght@400;700&display=swap" rel="stylesheet">
    <style>
        body {{
            direction: rtl;
            font-family: 'Noto Naskh Arabic', Arial, sans-serif;
            margin: 20px;
            background: #f5f5f5;
            line-height: 2;
        }}
        .container {{
            max-width: 1000px;
            margin: 0 auto;
            background: white;
            padding: 30px;
            border-radius: 8px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        }}
        h1 {{ text-align: center; color: #333; }}
        .info {{ text-align: center; color: #666; padding: 10px; background: #f0f0f0; border-radius: 5px; margin: 20px 0; }}
        .stats {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 15px; margin: 20px 0; }}
        .stat {{ padding: 15px; background: #f9f9f9; border-left: 4px solid #ff8c00; border-radius: 3px; }}
        .stat-label {{ font-size: 12px; color: #666; }}
        .stat-value {{ font-size: 20px; font-weight: bold; color: #ff8c00; }}
        .text {{ background: white; padding: 20px; border: 1px solid #ddd; border-radius: 5px; text-align: justify; margin: 20px 0; }}
        .boundary {{ background: rgba(255, 165, 0, 0.3); border-left: 2px solid #ff8c00; padding: 1px 2px; }}
        .footer {{ text-align: center; color: #999; font-size: 12px; padding-top: 20px; border-top: 1px solid #ddd; }}
    </style>
</head>
<body>
    <div class="container">
        <h1>📚 Boundary Tokens - Kitab Uqala</h1>
        <div class="info">Caractères surlignés en <span style="background: rgba(255, 165, 0, 0.3); padding: 2px 4px;">orange</span> = boundary tokens détectés par CAMeL-BERT</div>

        <div class="stats">
            <div class="stat">
                <div class="stat-label">Corpus</div>
                <div class="stat-value">{len(text):,}</div>
                <div class="stat-label">caractères</div>
            </div>
            <div class="stat">
                <div class="stat-label">Tokens</div>
                <div class="stat-value">{len(token_ids):,}</div>
                <div class="stat-label">tokens total</div>
            </div>
            <div class="stat">
                <div class="stat-label">Boundary Tokens</div>
                <div class="stat-value">{len(boundary_indices):,}</div>
                <div class="stat-label">tokens détectés</div>
            </div>
            <div class="stat">
                <div class="stat-label">Pourcentage</div>
                <div class="stat-value">{100 * len(boundary_indices) / len(token_ids):.2f}%</div>
                <div class="stat-label">du corpus</div>
            </div>
        </div>

        <div class="text">
            {html_text}
        </div>

        <div class="footer">
            <p>Source: camelbert_boundary_tokens_clean.json</p>
            <p>Model: CAMeL-BERT (camelbert_binary_classification_final)</p>
            <p>Generated: 2026-04-21</p>
        </div>
    </div>
</body>
</html>"""

# Save
with open('results/visualization_boundary_tokens_correct.html', 'w', encoding='utf-8') as f:
    f.write(html)

print(f"\n✓ HTML saved: results/visualization_boundary_tokens_correct.html")

In [ ]:
from google.colab import files
files.download('results/visualization_boundary_tokens_correct.html')
print("Downloaded!")